# LongMemEval-S · nautilus-compass v2.0.0 benchmark · Colab T4

Reproduces the LongMemEval-S accuracy run (paper baseline 56.6% with m3-rerank) on free Colab T4 GPU. ~6-10h walltime for full 500 questions.

## Before you run
1. Runtime → Change runtime type → T4 GPU
2. Upload your Gemini service account JSON to Colab (Files panel left). Name it `gemini-sa.json`.
3. Run cells top-to-bottom. Cell 5 is the long one (subset first to validate, then `--full`).

## Output
- `/content/longmemeval_compass_v2_results.json` — per-question hits + judge verdicts
- Final cell prints accuracy by question type.

Resume support: the runner skips questions already in the results file, so re-running picks up where it left off after a disconnect.

## 1. Verify T4 GPU

In [ ]:
!nvidia-smi | head -20

## 2. Install nautilus-compass v2.0.0 + dependencies

In [ ]:
!pip install -q nautilus-compass==2.0.0 sentence-transformers google-genai datasets
!pip install -q FlagEmbedding peft  # for bge-reranker-v2-m3 cross-encoder
import nautilus_compass
print('compass version:', nautilus_compass.__version__)

## 3. Download LongMemEval-S dataset (HuggingFace)

In [ ]:
import os, json
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = 'https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json'
DATA_PATH = Path('/content/longmemeval_s.json')
if not DATA_PATH.exists():
    urlretrieve(DATA_URL, DATA_PATH)
dataset = json.loads(DATA_PATH.read_text())
print(f'Loaded {len(dataset)} questions')
qt_counts = {}
for item in dataset:
    qt = item.get('question_type', 'unknown')
    qt_counts[qt] = qt_counts.get(qt, 0) + 1
print('by type:', qt_counts)

## 4. Configure Gemini Flash (subject + judge)

Before running this cell, upload your service account JSON to `/content/gemini-sa.json`.

In [ ]:
import os
SA_PATH = '/content/gemini-sa.json'
assert Path(SA_PATH).exists(), 'Upload gemini-sa.json to /content/ first (Files panel left)'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = SA_PATH
os.environ['COMPASS_USE_GEMINI_FLASH'] = '1'

# Verify Gemini Flash works
from nautilus_compass.judges.gemini_flash import GeminiFlashJudge
judge = GeminiFlashJudge()
resp = judge.generate('Say OK if you can read this.', max_tokens=10)
print('Gemini Flash test:', resp)

## 5. Load BGE-m3 embedder + reranker (GPU)

First load is slow (~3-5min download). Cached for subsequent runs.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

embedder = SentenceTransformer('BAAI/bge-m3', device=device)
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device, max_length=512)
print('models loaded')

## 6. Benchmark runner (resumable)

Per question:
1. Concat each history session into a single text chunk
2. BGE-m3 dense retrieve top-50
3. bge-reranker-v2-m3 rerank to top-5
4. Gemini Flash answer with top-5 as context
5. Gemini Flash judge (correct/wrong vs reference)

Set `SUBSET = 30` first to validate end-to-end (~30min). Then set `SUBSET = None` for full 500.

In [ ]:
import time
import json
from pathlib import Path
import numpy as np

SUBSET = 30  # set to None for full 500
TOP_K_RETRIEVE = 50
TOP_K_CONTEXT = 5
RESULTS_PATH = Path('/content/longmemeval_compass_v2_results.json')

# Resume support
results = []
if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text())
    print(f'Resuming from {len(results)} done')
done_qids = {r['question_id'] for r in results}

items = dataset if SUBSET is None else dataset[:SUBSET]

SUBJECT_PROMPT = '''You are a memory-augmented assistant. The retrieval system has filtered to {k} most relevant past sessions — the answer IS in them.

RULES:
1. Lead with the specific fact. NO preamble.
2. Maximum 1-2 sentences. Be concrete (number, name, date).
3. Only refuse if sessions are completely unrelated to the question.

=== Top {k} sessions ===
{context}

=== Question ===
{question}

Answer:'''

JUDGE_PROMPT = '''Judge whether the candidate answer correctly addresses the question. The reference answer is the ground truth.

Question: {question}
Reference: {reference}
Candidate: {candidate}

Respond with JSON only: {{"correct": true|false, "reason": "one short sentence"}}'''

t_start = time.time()
for idx, item in enumerate(items):
    qid = item['question_id']
    if qid in done_qids:
        continue

    question = item['question']
    reference = item.get('answer', '')
    qtype = item.get('question_type', 'unknown')
    sessions = item.get('haystack_sessions', [])

    # Concat each session into a single text chunk
    chunks = []
    for s in sessions:
        if isinstance(s, list):
            text = '\n'.join(f"{m.get('role','user')}: {m.get('content','')}" for m in s if isinstance(m, dict))
        else:
            text = str(s)
        chunks.append(text[:4000])  # cap per-session length

    if not chunks:
        results.append({'question_id': qid, 'hypothesis': '', 'judge_correct': False, 'judge_reason': 'no history', 'question_type': qtype})
        continue

    # 1. Dense retrieve
    q_emb = embedder.encode([question], normalize_embeddings=True)
    c_embs = embedder.encode(chunks, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    scores = (c_embs @ q_emb.T).flatten()
    top_dense_idx = np.argsort(-scores)[:TOP_K_RETRIEVE]

    # 2. Rerank
    pairs = [(question, chunks[i]) for i in top_dense_idx]
    rerank_scores = reranker.predict(pairs, batch_size=8, show_progress_bar=False)
    reranked_order = np.argsort(-rerank_scores)[:TOP_K_CONTEXT]
    top_chunk_indices = [top_dense_idx[i] for i in reranked_order]
    context = '\n\n---\n\n'.join(chunks[i][:2000] for i in top_chunk_indices)

    # 3. Gemini Flash answer
    subj_prompt = SUBJECT_PROMPT.format(k=TOP_K_CONTEXT, context=context, question=question)
    hypothesis = judge.generate(subj_prompt, max_tokens=300) or '[no answer]'

    # 4. Gemini Flash judge
    jud_prompt = JUDGE_PROMPT.format(question=question, reference=reference, candidate=hypothesis)
    verdict_raw = judge.generate(jud_prompt, max_tokens=200) or ''
    correct = False
    reason = 'parse error'
    try:
        v_clean = verdict_raw.strip()
        if v_clean.startswith('```'):
            v_clean = v_clean.split('```')[1]
            if v_clean.startswith('json'):
                v_clean = v_clean[4:].strip()
        v_obj = json.loads(v_clean)
        correct = bool(v_obj.get('correct', False))
        reason = v_obj.get('reason', '')[:200]
    except Exception as e:
        reason = f'parse: {e}'

    results.append({
        'question_id': qid,
        'question_type': qtype,
        'hypothesis': hypothesis,
        'judge_correct': correct,
        'judge_reason': reason,
        'top_indices': [int(i) for i in top_chunk_indices],
    })

    # Checkpoint every 5
    if (idx + 1) % 5 == 0:
        RESULTS_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False))
        elapsed = time.time() - t_start
        rate = (idx + 1 - len(done_qids)) / elapsed if elapsed > 0 else 0
        remaining = len(items) - len(results)
        eta_min = remaining / rate / 60 if rate > 0 else float('inf')
        print(f'  [{len(results)}/{len(items)}] {qid} → {"OK" if correct else "NO"} · rate={rate*60:.1f}/min · ETA={eta_min:.0f}min')

RESULTS_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f'\nDone. Wrote {len(results)} results to {RESULTS_PATH}')

## 7. Summary · accuracy by question type

In [ ]:
from collections import defaultdict
by_type = defaultdict(lambda: {'n': 0, 'ok': 0})
for r in results:
    t = r.get('question_type', 'unknown')
    by_type[t]['n'] += 1
    if r.get('judge_correct'):
        by_type[t]['ok'] += 1

print(f'{"qtype":<28} {"acc":>8} {"n":>5}')
print('-' * 45)
total_n = total_ok = 0
for t, d in sorted(by_type.items()):
    acc = d['ok'] / d['n'] if d['n'] else 0
    print(f'{t:<28} {acc:>7.1%} {d["n"]:>5}')
    total_n += d['n']
    total_ok += d['ok']
print('-' * 45)
print(f'{"OVERALL":<28} {total_ok/total_n if total_n else 0:>7.1%} {total_n:>5}')

## 8. Download results

Right-click `longmemeval_compass_v2_results.json` in the Files panel → Download.